# Traffic pattern matrices

Run the cell below to generate `traffic_patterns.png` with all-to-all, hot-cold, and mixed locality traffic matrices.

In [ ]:
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import matplotlib.pyplot as plt


# Edit these parameters as needed.
n_tors = 108
theta = 0.30
phi = 0.60
n_groups = 18
output_path = "traffic_patterns.png"
grayscale_output_path = "traffic_patterns_grayscale.png"
title_fontsize = 22
label_fontsize = 20
colorbar_fontsize = 20
colorbar_tick_fontsize = 18


def all_to_all_matrix(n_tors):
    return np.ones((n_tors, n_tors), dtype=float)


def hot_cold_matrix(n_tors, theta, phi):
    if not (0 < theta < 1 and 0 < phi < 1):
        raise ValueError("theta and phi must both be in (0, 1)")

    n_hot = int(np.ceil(theta * n_tors))
    matrix = np.zeros((n_tors, n_tors), dtype=float)

    hot_to_hot = (phi / theta) ** 2
    cold_to_cold = ((1 - phi) / (1 - theta)) ** 2
    hot_to_cold = (phi / theta) * ((1 - phi) / (1 - theta))

    matrix[:n_hot, :n_hot] = hot_to_hot
    matrix[n_hot:, n_hot:] = cold_to_cold
    matrix[:n_hot, n_hot:] = hot_to_cold
    matrix[n_hot:, :n_hot] = hot_to_cold
    return matrix


def mixed_locality_matrix(n_tors, n_groups):
    if n_tors % n_groups != 0:
        raise ValueError("n_tors must be divisible by n_groups")

    group_size = n_tors // n_groups
    matrix = np.zeros((n_tors, n_tors), dtype=float)
    for group in range(n_groups):
        start = group * group_size
        stop = start + group_size
        matrix[start:stop, start:stop] = 1.0
    return matrix


matrices = [
    all_to_all_matrix(n_tors),
    hot_cold_matrix(n_tors, theta, phi),
    mixed_locality_matrix(n_tors, n_groups),
]

titles = [
    "All-to-all",
    f"Hot-cold ($\u03B8$={theta:.1f}, $\u03C6$={phi:.1f})",
    f"Mixed locality ({n_groups} groups)",
]

display_matrices = [matrix / matrix.max() for matrix in matrices]

def plot_traffic_patterns(cmap, output_path, colorbar_label):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    fig.subplots_adjust(wspace=0.46, right=0.90, top=0.84)

    for ax, matrix, title in zip(axes, display_matrices, titles):
        image = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(title, fontsize=title_fontsize, pad=10)
        ax.set_xlabel("Destination ToR", fontsize=label_fontsize, labelpad=10)
        ax.set_ylabel("Source ToR", fontsize=label_fontsize, labelpad=15)
        ax.set_xticks([])
        ax.set_yticks([])

    colorbar = fig.colorbar(image, ax=axes, shrink=0.82, label=colorbar_label)
    colorbar.ax.yaxis.label.set_size(colorbar_fontsize)
    colorbar.ax.tick_params(labelsize=colorbar_tick_fontsize)
    colorbar.ax.yaxis.labelpad = 15
    fig.savefig(output_path, dpi=500, bbox_inches="tight")
    plt.show()


plot_traffic_patterns("viridis", output_path, "Normalized traffic weight")
plot_traffic_patterns("gray_r", grayscale_output_path, "Normalized traffic weight")

print(f"Saved {output_path}")
print(f"Saved {grayscale_output_path}")
